# 05 · Evaluación, interpretabilidad y prototipo
Guía 20 a 22. **Todas las cifras y figuras salen de `results/` (probabilidades guardadas por el notebook 04); no se escribe ningún número a mano.**

Modelos comparados: `dummy`, `logreg_color`, `cnn_scratch`, `resnet50` y `resnet50_deg` (variante con degradación de resolución, ya descartada por validación en el notebook 04; se incluye para documentarla).

**Reglas de este notebook**
- La **selección** de modelo y de semilla se hace solo con **validación**.
- El **test** y el **conjunto externo** solo se usan para reportar.
- Los recortes de una misma foto están correlacionados, así que además del bootstrap clásico se calcula un **bootstrap por grupos (foto de origen)**, que da intervalos más honestos.

## 0 · Entorno y carga

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import sys, os, json, glob, time
PROJECT = '/content/drive/MyDrive/proyecto_cafe_cnn'
os.environ['PROJECT_DIR'] = PROJECT
sys.path.insert(0, f'{PROJECT}/src')
import common as C
import extras
import numpy as np, pandas as pd, matplotlib.pyplot as plt, seaborn as sns
import tensorflow as tf
P = C.paths()
SEED = 42
C.set_seed(SEED)
plt.rcParams.update({'font.family': 'serif', 'figure.dpi': 120})

df = extras.prepare_local()                                  # datos locales, mismo orden que en el entrenamiento
classes = json.load(open(f"{P['data/processed']}/classes.json"))
BASE, DEG = f"{PROJECT}/results", f"{PROJECT}/variants/deg/results"
MODELS = {'dummy': ('dummy', BASE), 'logreg_color': ('logreg_color', BASE),
          'cnn_scratch': ('cnn_scratch', BASE), 'resnet50': ('resnet50', BASE),
          'resnet50_deg': ('resnet50', DEG)}

def seeds_of(name):
    kind, d = MODELS[name]
    return sorted(int(p.split('seed')[-1]) for p in glob.glob(f"{d}/{kind}_seed*") if os.path.isdir(p))
def load(name, seed, split):
    kind, d = MODELS[name]
    r = f"{d}/{kind}_seed{seed}"
    return np.load(f'{r}/y_{split}.npy'), np.load(f'{r}/probs_{split}.npy')

va = df[df.split == 'val'].reset_index(drop=True)
te = df[df.split == 'test'].reset_index(drop=True)
ex = df[(df.split == 'external') & df.label.isin(classes)].reset_index(drop=True)
for n in MODELS: print(f'{n:14s} semillas: {seeds_of(n)}')
for nm, d in [('val', va), ('test', te), ('external', ex)]:
    y, _ = load('resnet50', 42, nm); assert len(y) == len(d), f'{nm}: orden/tamaño no coincide'
print('Alineación de resultados y datos verificada')

## 1 · Métricas por corrida

In [ ]:
from sklearn.metrics import precision_recall_fscore_support as prfs
rows, prow = [], []
for name in MODELS:
    for s in seeds_of(name):
        for split in ['val', 'test', 'external']:
            try: y, p = load(name, s, split)
            except FileNotFoundError: continue
            rows.append(dict(model=name, seed=s, split=split, **C.compute_metrics(y, p)))
            pr, rc, f1, sup = prfs(y, p.argmax(1), labels=range(len(classes)), zero_division=0)
            for i, c in enumerate(classes):
                prow.append(dict(model=name, seed=s, split=split, clase=c, precision=pr[i], recall=rc[i], f1=f1[i], soporte=int(sup[i])))
res, pc = pd.DataFrame(rows), pd.DataFrame(prow)
res.to_csv(f"{P['results']}/metricas_por_corrida.csv", index=False)
pc.to_csv(f"{P['results']}/metricas_por_clase.csv", index=False)
res.head()

## 2 · Tabla comparativa (media ± desviación entre semillas)

In [ ]:
metrics = ['accuracy', 'balanced_accuracy', 'macro_f1', 'roc_auc_ovr']
tab = res.groupby(['split', 'model'])[metrics].agg(['mean', 'std']).round(3)
tab.to_csv(f"{P['results']}/tabla_comparativa.csv")
order = ['dummy', 'logreg_color', 'cnn_scratch', 'resnet50', 'resnet50_deg']
compact = (res.groupby(['model', 'split']).macro_f1.agg(['mean', 'std']).round(3)
              .apply(lambda r: f"{r['mean']:.3f} ± {0 if pd.isna(r['std']) else r['std']:.3f}", axis=1).unstack()
              .loc[order, ['val', 'test', 'external']])
display(compact)
display(tab.loc[['val', 'test', 'external']])

In [ ]:
# Fig. · Macro-F1 por modelo y partición (media ± desviación entre semillas)
g = res.groupby(['model', 'split']).macro_f1.agg(['mean', 'std']).fillna(0).reset_index()
fig, ax = plt.subplots(figsize=(9, 4.5)); w = 0.26
for k, sp in enumerate(['val', 'test', 'external']):
    d = g[g.split == sp].set_index('model').loc[order]
    ax.bar(np.arange(len(order)) + (k - 1) * w, d['mean'], w, yerr=d['std'], capsize=3, edgecolor='black', label=sp)
ax.set_xticks(range(len(order))); ax.set_xticklabels(order, rotation=20)
ax.set_ylabel('Macro-F1'); ax.set_title('Comparación de modelos'); ax.legend(); ax.grid(axis='y', ls='--', alpha=.5)
plt.tight_layout(); plt.savefig(f"{P['figures']}/comparacion_modelos.png", dpi=300); plt.show()

In [ ]:
# Recall por clase en test (media entre semillas): ¿qué clases fallan?
rec = pc[pc.split == 'test'].groupby(['model', 'clase']).recall.mean().unstack().loc[order, classes].round(3)
display(rec)
sop = pc[(pc.split == 'test') & (pc.model == 'resnet50') & (pc.seed == 42)].set_index('clase').soporte
print('Soporte en test:', sop.to_dict())
rec.to_csv(f"{P['results']}/recall_por_clase_test.csv")

## 3 · Selección del modelo (solo con validación)
Para cada modelo se toma la semilla con mayor macro-F1 en **validación**. El mejor modelo entre CNN desde cero y ResNet50 es el de mayor macro-F1 **medio** en validación.

In [ ]:
val = res[res.split == 'val']
sel_seed = {n: int(val[val.model == n].sort_values('macro_f1', ascending=False).iloc[0].seed) for n in MODELS}
best_name = max(['cnn_scratch', 'resnet50'], key=lambda n: val[val.model == n].macro_f1.mean())
best_seed = sel_seed[best_name]
print('Semilla elegida por modelo (validación):', sel_seed)
print(f'Mejor modelo por validación: {best_name} (semilla {best_seed})')

## 4 · Matriz de confusión y curvas ROC (test, mejor modelo)

In [ ]:
from sklearn.metrics import confusion_matrix, classification_report, roc_curve, auc
y, p = load(best_name, best_seed, 'test')
cm = confusion_matrix(y, p.argmax(1), labels=range(len(classes)))
fig, ax = plt.subplots(1, 2, figsize=(12, 4.5))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=classes, yticklabels=classes, ax=ax[0], cbar=False)
sns.heatmap(cm / cm.sum(1, keepdims=True), annot=True, fmt='.2f', cmap='Blues', xticklabels=classes, yticklabels=classes, ax=ax[1], cbar=False)
ax[0].set_title(f'Conteos (test) · {best_name}'); ax[1].set_title('Normalizada por clase real')
for a in ax: a.set_xlabel('Predicha'); a.set_ylabel('Real')
plt.tight_layout(); plt.savefig(f"{P['figures']}/matriz_confusion_test.png", dpi=300); plt.show()
print(classification_report(y, p.argmax(1), target_names=classes, digits=3))

# Principales confusiones (fuera de la diagonal)
cn = cm / cm.sum(1, keepdims=True)
conf = [(classes[i], classes[j], int(cm[i, j]), round(float(cn[i, j]), 3)) for i in range(len(classes)) for j in range(len(classes)) if i != j and cm[i, j] > 0]
display(pd.DataFrame(conf, columns=['real', 'predicha', 'n', 'fraccion_de_la_clase']).sort_values('fraccion_de_la_clase', ascending=False).head(6))

In [ ]:
# Curvas ROC uno-contra-resto calculadas con las probabilidades guardadas
fig, ax = plt.subplots(figsize=(6, 5))
for i, c in enumerate(classes):
    fpr, tpr, _ = roc_curve((y == i).astype(int), p[:, i]); ax.plot(fpr, tpr, label=f'{c} (AUC={auc(fpr, tpr):.3f})')
ax.plot([0, 1], [0, 1], '--', color='gray'); ax.set_xlabel('Tasa de falsos positivos'); ax.set_ylabel('Tasa de verdaderos positivos')
ax.set_title(f'ROC (test) · {best_name}'); ax.legend(); ax.grid(ls='--', alpha=.4)
plt.tight_layout(); plt.savefig(f"{P['figures']}/roc_test.png", dpi=300); plt.show()

## 5 · Incertidumbre y significancia estadística
- **IC 95 % del macro-F1:** bootstrap clásico sobre recortes y **bootstrap por grupos** (se remuestrean fotos de origen, no recortes sueltos).
- **Comparación pareada:** diferencia de macro-F1 con IC por grupos y prueba de McNemar sobre las mismas imágenes.

In [ ]:
from sklearn.metrics import f1_score
f1m = lambda yt, yp: f1_score(yt, yp, average='macro', labels=range(len(classes)), zero_division=0)

def cluster_boot(y, preds, groups, n=1000, seed=SEED):
    # Remuestrea grupos completos; devuelve {modelo: array de macro-F1} con los mismos remuestreos
    rng = np.random.default_rng(seed)
    uniq, inv = np.unique(groups, return_inverse=True)
    members = [np.where(inv == k)[0] for k in range(len(uniq))]
    out = {k: [] for k in preds}
    for _ in range(n):
        idx = np.concatenate([members[k] for k in rng.integers(0, len(uniq), len(uniq))])
        for k, pr in preds.items(): out[k].append(f1m(y[idx], pr[idx]))
    return {k: np.array(v) for k, v in out.items()}

ci = lambda v: (np.percentile(v, 2.5), np.percentile(v, 97.5))
names = ['logreg_color', 'cnn_scratch', 'resnet50', 'resnet50_deg']
y_te = load('resnet50', 42, 'test')[0]
pred_te = {n: load(n, sel_seed[n], 'test')[1].argmax(1) for n in names}
cb = cluster_boot(y_te, pred_te, te.group.values)
rows = []
for n in names:
    lo, hi = C.bootstrap_ci(y_te, pred_te[n]); clo, chi = ci(cb[n])
    rows.append(dict(modelo=n, semilla=sel_seed[n], macro_f1_test=round(f1m(y_te, pred_te[n]), 3),
                     ic95_clasico=f'[{lo:.3f}, {hi:.3f}]', ic95_por_grupos=f'[{clo:.3f}, {chi:.3f}]'))
ic_tab = pd.DataFrame(rows); display(ic_tab)
ic_tab.to_csv(f"{P['results']}/ic_macro_f1_test.csv", index=False)

In [ ]:
# Comparaciones pareadas: mejor modelo vs. cada referencia
from statsmodels.stats.contingency_tables import mcnemar
comp = []
for other in [n for n in ['logreg_color', 'cnn_scratch', 'resnet50_deg'] if n != best_name]:
    d = cb[best_name] - cb[other]; lo, hi = ci(d)
    a, b = pred_te[best_name] == y_te, pred_te[other] == y_te
    tt = [[(a & b).sum(), (a & ~b).sum()], [(~a & b).sum(), (~a & ~b).sum()]]
    disc = tt[0][1] + tt[1][0]
    pv = mcnemar(tt, exact=disc < 25, correction=True).pvalue
    comp.append(dict(comparacion=f'{best_name} vs {other}', dif_macro_f1=round(f1m(y_te, pred_te[best_name]) - f1m(y_te, pred_te[other]), 3),
                     ic95_dif_por_grupos=f'[{lo:.3f}, {hi:.3f}]', discordantes=int(disc), p_mcnemar=float(pv)))
comp = pd.DataFrame(comp); display(comp); comp.to_csv(f"{P['results']}/comparaciones_pareadas.csv", index=False)
print('Si el IC de la diferencia incluye 0, no hay evidencia de que los modelos difieran en macro-F1.')

## 6 · Validación externa (JMuBEN)
Mide la caída de desempeño al cambiar de país, cámara y condiciones. **Esta caída es un resultado**, no un fallo. Solo se evalúan las 4 clases comunes; `healthy` no existe en el entrenamiento (sección 6.3).

In [ ]:
metrics3 = ['accuracy', 'balanced_accuracy', 'macro_f1']
display(res[res.split == 'external'].groupby('model')[metrics3].agg(['mean', 'std']).round(3).loc[order])
piv = res.groupby(['model', 'split']).macro_f1.mean().unstack()[['val', 'test', 'external']].loc[order]
piv['caida_test_a_externa'] = piv['test'] - piv['external']
display(piv.round(3)); piv.round(3).to_csv(f"{P['results']}/caida_externa.csv")

y_ex = load('resnet50', 42, 'external')[0]
pred_ex = {n: load(n, sel_seed[n], 'external')[1].argmax(1) for n in ['cnn_scratch', 'resnet50']}
cbe = cluster_boot(y_ex, pred_ex, ex.group.values)
for n in pred_ex: print(f'{n:12s} macro-F1 externo = {f1m(y_ex, pred_ex[n]):.3f}  IC95% por grupos = [{ci(cbe[n])[0]:.3f}, {ci(cbe[n])[1]:.3f}]')

In [ ]:
# Matriz de confusión externa y recall por clase (mejor modelo)
yE, pE = load(best_name, best_seed, 'external')
cmE = confusion_matrix(yE, pE.argmax(1), labels=range(len(classes)))
fig, ax = plt.subplots(1, 2, figsize=(12, 4.5))
sns.heatmap(cmE, annot=True, fmt='d', cmap='Oranges', xticklabels=classes, yticklabels=classes, ax=ax[0], cbar=False)
sns.heatmap(cmE / cmE.sum(1, keepdims=True), annot=True, fmt='.2f', cmap='Oranges', xticklabels=classes, yticklabels=classes, ax=ax[1], cbar=False)
ax[0].set_title(f'Conteos (externo) · {best_name}'); ax[1].set_title('Normalizada por clase real')
for a in ax: a.set_xlabel('Predicha'); a.set_ylabel('Real')
plt.tight_layout(); plt.savefig(f"{P['figures']}/matriz_confusion_externa.png", dpi=300); plt.show()
print(classification_report(yE, pE.argmax(1), target_names=classes, digits=3))

In [ ]:
# 6.3 · ¿Qué hace el modelo con hojas sanas? (no es una métrica: ilustra una limitación)
model, base = C.build_resnet(len(classes))
model.load_weights(f"{P['models']}/{MODELS[best_name][0]}_seed{best_seed}.weights.h5") if best_name == 'resnet50' else None
if best_name == 'resnet50':
    hl = df[(df.split == 'external') & (df.label == 'healthy')].reset_index(drop=True)
    dsH = C.make_dataset(hl.path, np.zeros(len(hl), dtype=int), 64, False)
    predH = model.predict(dsH, verbose=0).argmax(1)
    print('Hojas sanas de JMuBEN (', len(hl), ') clasificadas por el modelo como:')
    display(pd.Series([classes[i] for i in predH]).value_counts().rename('n').to_frame())
else:
    print('El mejor modelo es la CNN desde cero; adapta esta celda para cargar sus pesos.')

## 7 · Costo computacional
Parámetros, tiempo de entrenamiento y épocas realmente entrenadas por modelo (leído de `history.json`).

In [ ]:
rows = []
for n in ['cnn_scratch', 'resnet50', 'resnet50_deg']:
    kind, d = MODELS[n]
    for s in seeds_of(n):
        h = json.load(open(f"{d}/{kind}_seed{s}/history.json"))
        rows.append(dict(modelo=n, semilla=s, parametros=h['params'], minutos=round(h['train_seconds'] / 60, 1),
                         epocas=sum(len(ph['loss']) for ph in h['history'])))
cost = pd.DataFrame(rows); display(cost)
display(cost.groupby('modelo')[['parametros', 'minutos', 'epocas']].mean().round(1))
cost.to_csv(f"{P['results']}/costo_computacional.csv", index=False)

In [ ]:
# Latencia de inferencia de una sola imagen (requisito RNF2) con el modelo ResNet50 cargado
if best_name == 'resnet50':
    x1 = np.zeros((1, 224, 224, 3), dtype=np.float32)
    for _ in range(3): model(x1, training=False)                       # calentamiento
    t0 = time.time()
    for _ in range(30): model(x1, training=False)
    ms = (time.time() - t0) / 30 * 1000
    print(f'Latencia por imagen (predicción): {ms:.1f} ms en {"GPU" if tf.config.list_physical_devices("GPU") else "CPU"}')

## 8 · Interpretabilidad (Grad-CAM) y análisis de errores

In [ ]:
def show_cam(frame, y_true, pred, title, fname, k_ok=3, k_bad=3):
    rng = np.random.default_rng(SEED)
    ok, bad = np.where(pred == y_true)[0], np.where(pred != y_true)[0]
    sel = list(rng.choice(ok, k_ok, replace=False)) + (list(rng.choice(bad, min(k_bad, len(bad)), replace=False)) if len(bad) else [])
    fig, axes = plt.subplots(2, len(sel), figsize=(2.8 * len(sel), 6))
    for k, i in enumerate(sel):
        img = np.array(Image.open(frame.path[i]).convert('RGB').resize((224, 224)))
        cam, c, probs = C.gradcam(model, img)
        axes[0, k].imshow(img); axes[0, k].axis('off')
        axes[0, k].set_title(f'real: {classes[y_true[i]]}\npred: {classes[pred[i]]}', fontsize=9, color='green' if pred[i] == y_true[i] else 'crimson')
        axes[1, k].imshow(img); axes[1, k].imshow(cam, cmap='jet', alpha=.4); axes[1, k].axis('off')
    plt.suptitle(title); plt.tight_layout(); plt.savefig(f"{P['figures']}/{fname}", dpi=300); plt.show()

from PIL import Image
if best_name == 'resnet50':
    yT, pT = load('resnet50', best_seed, 'test')
    show_cam(te, yT, pT.argmax(1), 'Grad-CAM · test (BRACOL): 3 aciertos y 3 errores', 'gradcam_test.png')
    yX, pX = load('resnet50', best_seed, 'external')
    show_cam(ex, yX, pX.argmax(1), 'Grad-CAM · externo (JMuBEN): 3 aciertos y 3 errores', 'gradcam_externo.png')

**Cómo leer estas figuras.** Un buen modelo concentra el calor sobre la lesión; si el calor cae en bordes, fondo blanco o gris, o en la esquina del recorte, hay indicio de atajo (el EDA ya mostró que algunos recortes de BRACOL incluyen fondo). Anota 2–3 observaciones concretas para la discusión del informe.

## 9 · Prototipo (Gradio) — guía 20.5 y 20.6
Se ejecuta en Colab y genera un enlace público temporal. **Entrada esperada:** un recorte de lesión, como los usados en el entrenamiento; con una foto de hoja completa el resultado no está validado.

In [ ]:
!pip -q install gradio

In [ ]:
import gradio as gr
def predict(img):
    if img is None: return None, None
    arr = np.array(img.convert('RGB').resize((224, 224)), dtype=np.float32)
    cam, c, probs = C.gradcam(model, arr)
    heat = plt.get_cmap('jet')(cam)[..., :3]
    overlay = (0.6 * arr / 255 + 0.4 * heat).clip(0, 1)
    return {cl: float(p_) for cl, p_ in zip(classes, probs)}, (overlay * 255).astype('uint8')

demo = gr.Interface(fn=predict, inputs=gr.Image(type='pil', label='Recorte de lesión en hoja de cafeto'),
                    outputs=[gr.Label(num_top_classes=len(classes), label='Diagnóstico estimado'), gr.Image(label='Grad-CAM')],
                    title='Clasificador de enfermedades foliares del cafeto',
                    description='Prototipo de investigación. No sustituye el diagnóstico de un especialista.')
demo.launch(share=True)

## Qué copiar al informe
| Archivo | Sección del informe |
|---|---|
| `tabla_comparativa.csv`, `comparacion_modelos.png` | 21 (Resultados) |
| `matriz_confusion_test.png`, `roc_test.png`, `recall_por_clase_test.csv` | 21 |
| `ic_macro_f1_test.csv`, `comparaciones_pareadas.csv` | 21 (incertidumbre y significancia) |
| `caida_externa.csv`, `matriz_confusion_externa.png` | 21 y 23 (validez externa) |
| `gradcam_test.png`, `gradcam_externo.png` | 22 (Discusión) |
| `costo_computacional.csv` y latencia | 20 (evaluación del artefacto) |
| `decision_degradacion.json` (notebook 04) | 19 y 21 (variante A/B) |
| `requirements_freeze.txt` | 25 (Reproducibilidad) |